In [1]:
import sqlite3
#connecting to the database 
connection = sqlite3.connect("library.db")
cursor = connection.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS books(   id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT UNIQUE NOT NULL,
        author TEXT NOT NULL,
        status TEXT DEFAULT 'available',
        borrowed_by TEXT
        )
    """)
connection.commit()



#add function to library.
def add_book():
    print("\n---Add book to library.---")

    title = input("Enter a book title: ").strip().title()
    if not title:
        print("❌ Book title cannot be empty.")
        return

    author = input("Enter the name of author: ").strip().title()
    if not author:
        print("❌ Author name cannot be empty.")
        return

    try:
        cursor.execute("""
            INSERT INTO books (title, author, status, borrowed_by)
            VALUES (?, ?, 'available', NULL)
        """, (title, author))
        connection.commit()
        print(f"✅ '{title}' by {author} added to library")

    except sqlite3.IntegrityError:
        print(f"❌ '{title}' already exists in library")
    except Exception as e:
        print(f"❌ Something went wrong: {e}")

#Display available books that are available to borrow
def show_available_books():
    print("\n----Available Books----")

    try:
        cursor.execute("SELECT title, author FROM books WHERE status = 'available'")
        available_books = cursor.fetchall()

        if not available_books:
            print("❌ No books available at the moment")
        else:
            print(f"Found {len(available_books)} available book(s):")
            for i, book in enumerate(available_books, 1):
                print(f" {i}. {book[0]} by {book[1]}")

        return available_books

    except Exception as e:
        print(f"❌ Something went wrong: {e}")
        return []  


#The borrow book function
def borrow_book():
    print("\n----Borrow a Book----")

    available_books = show_available_books()

    if not available_books:
        print("❌ Cannot borrow. No books available.")
        return

    title = input("Enter book title to borrow: ").strip().title()
    if not title:
        print("❌ Book title cannot be empty.")
        return

    borrower = input("Enter your name: ").strip().title()
    if not borrower:
        print("❌ Borrower name cannot be empty.")
        return

    try:
        # First check if the book exists and is available
        cursor.execute("SELECT status FROM books WHERE title = ?", (title,))
        book = cursor.fetchone()

        if not book:
            print(f"❌ '{title}' not found in library")
            return

        if book[0] != "available":
            print(f"❌ '{title}' is already borrowed")
            return

        # Update the book status
        cursor.execute("""
            UPDATE books
            SET status = 'borrowed', borrowed_by = ?
            WHERE title = ?
        """, (borrower, title))
        connection.commit()
        print(f"✅ '{title}' borrowed by {borrower}")

    except Exception as e:
        print(f"❌ Something went wrong: {e}")


#Return a book function.
def return_book():
    print("\n----Return a Book----")

    try:
        # Find all borrowed books
        cursor.execute("SELECT title, borrowed_by FROM books WHERE status = 'borrowed'")
        borrowed_books = cursor.fetchall()

        if not borrowed_books:
            print("❌ No books are currently borrowed")
            return

        print("\nCurrently borrowed books:")
        for book in borrowed_books:
            print(f" - {book[0]} (borrowed by {book[1]})")

        title = input("\nEnter the book title to return: ").strip().title()
        if not title:
            print("❌ Book title cannot be empty.")
            return

        # Check if this specific book is actually borrowed
        cursor.execute("SELECT status FROM books WHERE title = ?", (title,))
        book = cursor.fetchone()

        if not book:
            print(f"❌ '{title}' not found in library")
            return

        if book[0] != "borrowed":
            print(f"❌ '{title}' is not currently borrowed")
            return

        # Return the book
        cursor.execute("""
            UPDATE books
            SET status = 'available', borrowed_by = NULL
            WHERE title = ?
        """, (title,))
        connection.commit()
        print(f"✅ '{title}' has been returned successfully")

    except Exception as e:
        print(f"❌ Something went wrong: {e}")
#Saerch book function.
def search_books():
    print("\n---Search Books---")

    search_term = input("Enter book title or author to search: ").strip()
    if not search_term:
        print("❌ Search term cannot be empty.")
        return

    try:
        # Use LIKE for pattern matching - searches both title and author
        cursor.execute("""
            SELECT title, author, status, borrowed_by
            FROM books
            WHERE title LIKE ? OR author LIKE ?
        """, (f"%{search_term}%", f"%{search_term}%"))

        found_books = cursor.fetchall()

        if not found_books:
            print(f"❌ No books found matching '{search_term}'")
        else:
            print(f"\nFound {len(found_books)} book(s):")
            for book in found_books:
                title, author, status, borrowed_by = book
                if status == "available":
                    status_display = "✅ Available"
                else:
                    status_display = f"❌ Borrowed by {borrowed_by}"
                print(f"📚 '{title}' by {author} - {status_display}")

    except Exception as e:
        print(f"❌ Something went wrong: {e}")

#Show All books
def show_all_books():
    print("\n--- All Books in Library ---")

    try:
        cursor.execute("SELECT title, author, status, borrowed_by FROM books")
        all_books = cursor.fetchall()

        if not all_books:
            print("❌ Library is empty. Add some books first.")
        else:
            for book in all_books:
                title, author, status, borrowed_by = book
                if status == "available":
                    status_display = "✅ Available"
                else:
                    status_display = f"❌ Borrowed by {borrowed_by}"
                print(f"  📚 {title} by {author} - {status_display}")

    except Exception as e:
        print(f"❌ Something went wrong: {e}")

#Menu                                       
def main_menu():
    while True:
        print("\n" + "="*50)
        print("LIBRARY MANAGEMENT SYSTEM")
        print("="*50)
        print("1. Add a Book")
        print("2. Show Available Books")
        print("3. Borrow a Book")
        print("4. Return a Book")
        print("5. Search Books")
        print("6. Show All Books")
        print("7. Exit")
        print("="*50)

        choice = input("Enter your choice (1-7): ").strip()

        if choice == "1":
            add_book()              # ← no more my_library argument
        elif choice == "2":
            show_available_books()
        elif choice == "3":
            borrow_book()
        elif choice == "4":
            return_book()
        elif choice == "5":
            search_books()
        elif choice == "6":
            show_all_books()       
        elif choice == "7":
            connection.close()      # Close data base
            print("\n✅ Thank you for using the Library System. Goodbye!")
            break
        else:
            print("❌ Invalid choice. Please enter 1-7.")

# Run the program
main_menu()


LIBRARY MANAGEMENT SYSTEM
1. Add a Book
2. Show Available Books
3. Borrow a Book
4. Return a Book
5. Search Books
6. Show All Books
7. Exit


Enter your choice (1-7):  1



---Add book to library.---


Enter a book title:  Python crash course
Enter the name of author:  Javascripto


✅ 'Python Crash Course' by Javascripto added to library

LIBRARY MANAGEMENT SYSTEM
1. Add a Book
2. Show Available Books
3. Borrow a Book
4. Return a Book
5. Search Books
6. Show All Books
7. Exit


...


--- All Books in Library ---
  📚 Python Crash Course by Javascripto - ✅ Available
  📚 How To Be Rich In One Week by Robert Kivo Sakie - ✅ Available

LIBRARY MANAGEMENT SYSTEM
1. Add a Book
2. Show Available Books
3. Borrow a Book
4. Return a Book
5. Search Books
6. Show All Books
7. Exit


Enter your choice (1-7):  8


❌ Invalid choice. Please enter 1-7.

LIBRARY MANAGEMENT SYSTEM
1. Add a Book
2. Show Available Books
3. Borrow a Book
4. Return a Book
5. Search Books
6. Show All Books
7. Exit


Enter your choice (1-7):  7



✅ Thank you for using the Library System. Goodbye!


In [2]:
import os
print(os.getcwd())

C:\Users\hope
